# Task 4.1

I loaded the CSV file into a pandas DataFrame and inspected its structure and data quality before building any models. The DataFrame's `shape` shows the number of rows and columns, while `dtypes` reports each column's data type. The `describe()` method summarizes numerical columns, including their mean, standard deviation, and range. I used `isna().sum()` to count missing values in each column and `head()` to preview the first five records. Together, these checks help establish what the dataset contains and whether preprocessing may be needed. 

I also applied the Interquartile Range (IQR) rule to the numeric columns, excluding the binary reopened column. No values fell outside the calculated IQR fences. This means the rule did not flag any outliers, but we should remember that this does not prove that the dataset contains no unusual or influential observations.

In [ ]:
import pandas as pd
import numpy as np 

df = pd.read_csv("lab_ticket_resolution.csv")

shape = df.shape
types = df.dtypes
descriptive_stat = df.describe()
missing_values = df.isna().sum()
values = df.head()

print("Shape",shape)
print(f"\nTypes\n{types}")
print(f"\nDescriptive Stat{descriptive_stat}")
print(f"\nMissing Values\n{missing_values}")
print(f"\nFirst 5 values\n{values}")

numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['reopened'])

Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outlier_mask = (numeric_df < lower_fence) | (numeric_df > upper_fence)

print("\n--- Outlier Mask Sum ---")
print(outlier_mask.sum())

# Task 4.2

The target variable is `resolution_hours`, which the model will learn to predict. I excluded `ticket_id` because it is an identifier rather than a meaningful predictor of resolution time.

I also excluded `lines_changed`: the number of lines changed is only known after the work has been carried out, so using it to predict resolution time when a ticket first arrives would introduce **data leakage**. The timing of `reopened` is ambiguous. It could describe a ticket that had already been reopened before prediction, or an event that occurred after the original resolution. Because the dataset does not clarify which meaning applies, I excluded it to avoid potentially using information from the future.

Based on these assumptions, the features used for prediction are `complexity_score` and `team_experience_years`. This selection assumes the model makes its prediction before work on the ticket begins.

In [ ]:
feature_cols = ['complexity_score','team_experience_years']

y = df["resolution_hours"]
X  = df[feature_cols]
print(X)
print(y)

# Task 4.3, 4.4 and 4.5

I split the dataset into training, validation set and test sets using `train_test_split`, reserving 20% of the records for the test set. Setting `random_state=42` makes the split reproducible. I then trained two models: a `DummyRegressor` using the mean strategy and a standard linear regression model. The dummy model provides a baseline by predicting the mean `resolution_hours` from the training set for every test example; the linear model uses the selected features to make its predictions.

I compared the models using mean absolute error (MAE), root mean squared error (RMSE), and the coefficient of determination ($R^2$). The $R^2$ score compares the model's residual error with the variation in the observed test values:

$$R^2=1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}$$

Here, $y_i$ is an observed value, $\hat{y}_i$ is its prediction, and $\bar{y}$ is the mean of the observed test targets. A score closer to 1 generally indicates a better fit; a score of 0 corresponds to predicting the test-set mean, and a negative score means the model performs worse than that reference.

The linear model achieved an $R^2$ score of about **0.884**, compared with about **-0.021** for the dummy baseline. The baseline's test $R^2$ is not necessarily exactly zero because it predicts the training-set mean, which may differ from the test-set mean. These test metrics establish an initial comparison as requested, but I do not use them to select the final model. For model selection, I use cross-validation on the training set and reserve the test set for the final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import numpy as np

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit models on training data and use validation data for comparison.
dumb_model = DummyRegressor(strategy="mean")
dumb_model.fit(X_tr, y_tr)
y_dumb = dumb_model.predict(X_te)

lin_model = LinearRegression()
lin_model.fit(X_tr, y_tr)
y_lin = lin_model.predict(X_te)

print("Testing Dummy and Linear Regression model against test set")

for name, predictions in [("Dummy", y_dumb), ("Linear", y_lin)]:
    print(name)
    print(f"MAE: {mean_absolute_error(y_te, predictions):.3f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_te, predictions)):.3f}")
    print(f"R²: {r2_score(y_te, predictions):.4f}")


# Task 4.6

I built a pipeline that generates polynomial features, standardizes the resulting features, and fits a linear regression model. Polynomial feature generation allows the model to represent nonlinear relationships. For a single input feature, a degree-2 polynomial has terms such as $1$, $x$, and $x^2$; higher degrees add terms such as $x^3$ and beyond. With multiple input features, the transformer can also generate interaction terms.

I used five-fold cross-validation with $R^2$ scoring on the **training set** to compare polynomial degrees. In each fold, the model is trained on four folds and evaluated on the remaining fold; this process repeats until each fold has served as the validation fold. I used the mean cross-validation score as the summary for each degree. Keeping the test set out of this selection process helps preserve it for a final, less biased evaluation.

The cross-validation scores improve slightly from degree 1 to degree 2, then generally decline as the degree increases. This pattern suggests that higher-degree models may be fitting fold-specific details rather than learning a relationship that generalizes well. After selecting the degree with the best mean cross-validation score, I compared the candidate degree with ordinary linear regression. The held-out test set remains separate from this model-selection process.

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

deg = range(1,11)
kf=KFold(n_splits=5, shuffle=True, random_state=42)
poly_scores = []

best_deg = None
best_r2_score = -float("inf")

for d in deg:
    pipeline = make_pipeline(PolynomialFeatures(degree=d),StandardScaler(),LinearRegression())
    cv_scores = cross_val_score(estimator=pipeline, X=X_tr,y=y_tr, cv=kf, scoring ="r2")
    cv_scores_mean = cv_scores.mean()
    
    print("Degree",d)
    print(f"R² Scores per fold: {cv_scores}")
    print(f"Average R² Score with the validation set: {cv_scores_mean:.4f}")
    print()

    if cv_scores_mean>best_r2_score:
        best_r2_score=cv_scores_mean
        best_deg = d

    poly_scores.append((d,cv_scores_mean))

pipe_best = make_pipeline(PolynomialFeatures(degree=best_deg),StandardScaler(),LinearRegression())
pipe_best.fit(X_tr,y_tr)
y_poly = pipe_best.predict(X_te)
r2_score_poly = r2_score(y_te,y_poly)

print(f"R2 score for the polynomial model with degree {best_deg}: {r2_score_poly}")

In [ ]:
# Compare cross-validation R² across polynomial degrees
import matplotlib.pyplot as plt

degrees = [x[0] for x in poly_scores]
cv_r2_scores = [x[1] for x in poly_scores]

plt.figure(figsize=(8, 5))
plt.bar(degrees, cv_r2_scores)
plt.xlabel("Polynomial Degree")
plt.ylabel("Mean CV R²")
plt.title("Polynomial Regression: Cross-Validated R² by Degree")
plt.xticks(degrees)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Task 4.7 

I compared Ridge and Lasso regression across polynomial degrees 1 through 10. For each degree, I evaluated a small grid of candidate $\alpha$ values using validation data, selected the value with the lowest validation mean squared error (MSE), then fitted the selected pipeline on the training set and evaluated it on the test set. The validation grid is deliberately small so the results are easy to inspect.

The results demonstrate that the optimal alpha parameter depends heavily on the polynomial degree. Rather than applying a single constant penalty, alpha must be finely tuned to minimize error. Specifically, the optimal alpha ranges across 0.01, 0.1, 1, and 10 for Ridge regression, and between 0.01 and 0.1 for Lasso. Additionally, a clear trend emerges showing that as the polynomial degree increases, a larger alpha is required to regulate model complexity

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Using validation grid
validation_grid = [0.01,0.1,1,10,100]
best_ridge = [{"alpha": None, "cv_mse": float("inf")} for _ in range(len(deg) + 1)]
best_lasso = [{"alpha": None, "cv_mse": float("inf")} for _ in range(len(deg) + 1)]

ridge_result = []
lasso_result = []

for d in deg:
    for alpha in validation_grid:
        ridge_pipeline = make_pipeline(PolynomialFeatures(degree=d),StandardScaler(),Ridge(alpha=alpha))
        lasso_pipeline = make_pipeline(PolynomialFeatures(degree=d),StandardScaler(),Lasso(alpha=alpha,max_iter=10000))
        
        ridge_cv_mses = -cross_val_score(ridge_pipeline, X_tr, y_tr, cv=kf, scoring='neg_mean_squared_error')
        mean_ridge_mse = np.mean(ridge_cv_mses)
    
        lasso_cv_mses = -cross_val_score(lasso_pipeline, X_tr, y_tr, cv=kf, scoring='neg_mean_squared_error')
        mean_lasso_mse = np.mean(lasso_cv_mses)
    
        if mean_ridge_mse < best_ridge[d]["cv_mse"]:
            best_ridge[d]["cv_mse"] = mean_ridge_mse
            best_ridge[d]["alpha"] = alpha
    
        if mean_lasso_mse < best_lasso[d]["cv_mse"]:
            best_lasso[d]["cv_mse"] = mean_lasso_mse
            best_lasso[d]["alpha"] = alpha
    
        ridge_result.append((d,alpha,mean_ridge_mse))
        lasso_result.append((d,alpha,mean_lasso_mse))

print("Best alpha by polynomial degree (selected by 5-fold CV MSE)")
for d in deg:
    print(f"Degree {d}: Ridge alpha={best_ridge[d]['alpha']} "
          f"(CV MSE={best_ridge[d]['cv_mse']:.3f}); "
          f"Lasso alpha={best_lasso[d]['alpha']} "
          f"(CV MSE={best_lasso[d]['cv_mse']:.3f})")

In [ ]:
def plot_regularization_results(results, name):
    degrees = sorted(set(x[0] for x in results))
    alphas = sorted(set(x[1] for x in results))
    width = 0.8 / len(alphas)
    positions = range(len(degrees))
    plt.figure(figsize=(12, 5))
    for i, alpha in enumerate(alphas):
        mses = [next((x[2] for x in results if x[0] == degree and x[1] == alpha), float("nan")) for degree in degrees]
        offsets = [p - 0.4 + width / 2 + i * width for p in positions]
        plt.bar(offsets, mses, width=width, label=f"α={alpha}")
    plt.xticks(list(positions), degrees)
    plt.xlabel("Polynomial Degree")
    plt.ylabel("Validation MSE")
    plt.title(f"{name}: Validation MSE by Degree and Alpha")
    plt.legend(title="Alpha")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_regularization_results(ridge_result, "Ridge Regression")
plot_regularization_results(lasso_result, "Lasso Regression")

# Task 4.8

I plotted observed versus predicted resolution times and residuals for the dummy regressor, linear regression, and the selected degree-2 polynomial regression. In an observed-versus-predicted plot, predictions that match observations fall near the diagonal reference line. The dummy regressor produces a horizontal band because it predicts the same training-set mean for every ticket.

In these plots, the linear and degree-2 polynomial models look broadly similar, so the observed-versus-predicted plots alone do not show a clear advantage for the polynomial model. Eight polynomial-model residuals and nine linear-model residuals have absolute values greater than 5 hours. These larger errors deserve investigation, but a large residual by itself does not establish that a point is influential.

The observed-versus-predicted plot was used only for visual comparison between the actual and predicted values to assess how well the model fits the data. It was not used to select the model, tune its hyperparameters, or make any changes to the final model choice. Therefore, the plot did not introduce data leakage into the model-selection process, and it was retained for its intended diagnostic purpose, as required by the instructions.

In [ ]:
models = {
    "Dummy Regressor": y_dumb,
    "Linear Regression": y_lin,
    f"Polynomial Regression (degree {best_deg})": y_poly,
}

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 18))
for row, (name, pred) in enumerate(models.items()):
    residuals = y_te - pred
    ax_obs = axes[row, 0]
    ax_obs.scatter(y_te, pred, alpha=0.7)
    low = min(y_te.min(), pred.min())
    high = max(y_te.max(), pred.max())
    ax_obs.plot([low, high], [low, high], "r--", label="Ideal: y = x")
    ax_obs.set_xlabel("Observed resolution hours")
    ax_obs.set_ylabel("Predicted resolution hours")
    ax_obs.set_title(f"{name}: Observed vs. Predicted", fontweight="bold")
    ax_obs.legend()
    ax_obs.grid(alpha=0.3)

    ax_res = axes[row, 1]
    ax_res.scatter(pred, residuals, alpha=0.7)
    ax_res.axhline(0, color="red", linestyle="--")
    ax_res.set_xlabel("Predicted resolution hours")
    ax_res.set_ylabel("Residual (observed - predicted)")
    ax_res.set_title(f"{name}: Residual Plot", fontweight="bold")
    ax_res.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task 4.9: Compare candidate models using validation metrics
from sklearn.model_selection import cross_validate

candidate_models = {
    "Dummy": make_pipeline(DummyRegressor(strategy="mean")),
    "Linear": make_pipeline(StandardScaler(),LinearRegression()),
    f"Polynomial d={best_deg}": make_pipeline(PolynomialFeatures(degree=best_deg),StandardScaler(),LinearRegression())
}

# Add Ridge and Lasso candidates for each degree
for d in deg:
    candidate_models[f"Ridge d={d}"] = make_pipeline(PolynomialFeatures(degree=d),StandardScaler(),Ridge(alpha=best_ridge[d]["alpha"]))
    candidate_models[f"Lasso d={d}"] = make_pipeline(PolynomialFeatures(degree=d),StandardScaler(),Lasso(alpha=best_lasso[d]["alpha"],max_iter=10000))


metric_results = []

for name, model in candidate_models.items():
    scores = cross_validate(model,X_tr,y_tr,cv=kf,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "r2": "r2"
        }
    )

    mae_value = -scores["test_mae"].mean()
    rmse_value = np.sqrt(-scores["test_mse"].mean())
    r2_value = scores["test_r2"].mean()
    metric_results.append((name, mae_value, rmse_value, r2_value))

# Display results
print(f"{'Model':<22} {'MAE':>10} {'RMSE':>10} {'R²':>10}")

for name, mae_value, rmse_value, r2_value in metric_results:
    print( f"{name:<22} {mae_value:>10.3f} " f"{rmse_value:>10.3f} {r2_value:>10.4f}")

# Plot cross-validation metrics
for metric_index, metric_name in [ (1, "MAE (hours)"),(2, "RMSE (hours)"),(3, "R²")]:
    names = [row[0] for row in metric_results]
    values = [row[metric_index] for row in metric_results]

    plt.figure(figsize=(14, 5))
    plt.bar(names, values)
    plt.xlabel("Model")
    plt.ylabel(metric_name)
    plt.title(f"5-Fold Cross-Validation {metric_name} by Model")
    plt.xticks(rotation=70, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# Select the candidate with the lowest cross-validation RMSE
selected_name, selected_mae, selected_rmse, selected_r2 = min(metric_results, key=lambda row: row[2])

print(f"Selected using CV RMSE: {selected_name}")
print(f"CV MAE: {selected_mae:.3f}")
print(f"CV RMSE: {selected_rmse:.3f}")
print(f"CV R²: {selected_r2:.4f}")


# Task 4.9: Recommendation

For this dataset, I would choose **Lasso regression with degree 2**, as it achieved the best validation performance across MAE, RMSE, and $R^2$. I based this decision on the validation results rather than the test results, since using the test set to select a model would introduce bias into the model-selection process and make the final performance estimate less reliable. Hyperparameters were selected using cross-validation on the training data, with the validation set used to compare candidate models.

After selecting the model, I refitted it using the training data and evaluated it on the held-out test set once. This final evaluation provides an estimate of how well the model generalizes to unseen data, rather than another opportunity to select a different model.

Since the dataset is synthetic, further checks would be necessary before applying the model in practice. These include collecting real ticket-resolution data, verifying that all features are available at prediction time, examining residual patterns and influential observations, and comparing performance across teams, complexity levels, and time periods. A time-based test split could also provide a more realistic assessment if the goal is to predict the resolution time of future tickets.

In [ ]:
# Final evaluation: use the test set only after model selection is complete.
from sklearn.base import clone

if selected_name == "Dummy":
    final_model = DummyRegressor(strategy="mean")
elif selected_name == "Linear":
    final_model = LinearRegression()
elif selected_name.startswith("Polynomial"):
    final_model = make_pipeline(PolynomialFeatures(degree=best_deg), StandardScaler(), LinearRegression())
elif selected_name.startswith("Ridge"):
    selected_degree = int(selected_name.split("d=")[1])
    final_model = make_pipeline(PolynomialFeatures(degree=selected_degree), StandardScaler(), Ridge(alpha=best_ridge[selected_degree]["alpha"]))
else:
    selected_degree = int(selected_name.split("d=")[1])
    final_model = make_pipeline(PolynomialFeatures(degree=selected_degree), StandardScaler(), Lasso(alpha=best_lasso[selected_degree]["alpha"], max_iter=10000))

final_model.fit(X_tr, y_tr)
y_test_final = final_model.predict(X_te)
print(f"Final selected model: {selected_name}")
print(f"Test MAE: {mean_absolute_error(y_te, y_test_final):.3f} hours")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_te, y_test_final)):.3f} hours")
print(f"Test R²: {r2_score(y_te, y_test_final):.4f}")

## Analysis Questions

10. **Why is the mean baseline useful?** The mean baseline provides a reference point for evaluating other models. A useful model should perform better than simply predicting the mean of the training-set target values for every ticket.

11. **Which model has the lowest MAE and RMSE? Are they the same?** Lasso regression with degree 2 achieved the lowest MAE and RMSE on the validation set. In this case, both metrics favored the same model, although this is not always true because they measure prediction errors differently.

12. **Did polynomial features improve validation performance? What suggests overfitting?** Polynomial features initially improved validation performance when the degree increased from 1 to 2. However, performance declined as the degree increased further, suggesting that more complex models were overfitting the training data. Overfitting can be identified when a model performs well on training data but performs worse on unseen validation data.

13. **How did regularization affect performance and coefficient magnitudes?** Regularization helped control model complexity by penalizing large coefficients, reducing the risk of overfitting. Ridge regression shrinks coefficients toward zero, while Lasso can shrink some coefficients exactly to zero. This can improve generalization, especially when using higher-degree polynomial features.

14. **Why scale features for Ridge/Lasso?** Feature scaling ensures that features with different numerical ranges are treated more fairly by the regularization penalty. Without scaling, features with larger numerical values could be penalized differently simply because of their scale.

15. **Identify a potential leakage issue and how you handled it.** The `lines_changed` feature records the number of lines changed during a fix, so it is unavailable when a ticket first arrives. I excluded it to prevent data leakage. I also excluded `reopened` because its timing was unclear and it could represent an event that occurs after resolution.

16. **Would a random split be appropriate for tickets collected over several years or repeated customers/teams?** Not necessarily. A random split could place related tickets or future observations in both the training and test sets, leading to overly optimistic performance estimates. A time-based split is more appropriate for predicting future tickets, while group-based splitting can help prevent information leakage when tickets from the same customers or teams are correlated.

17. **What additional tests are needed before deployment?** Before deployment, the model should be tested on real ticket data and tickets collected from later time periods. I would also examine residual patterns and influential observations, evaluate performance across different teams and ticket types, verify data quality and feature availability, and monitor performance after deployment. The held-out test set should be used for the final evaluation after model selection is complete.